# Practice run for PreRun code

In [1]:
from PreRun_alt import PreRun
import pandas as pd
import numpy as np
import pyarrow.parquet as pq

In [2]:
systems_cleaned = pd.read_csv('../../../data/core/systems_cleaned.csv')

In [3]:
my_rough_data_pq = pq.ParquetDataset('./temp_fix/other/4/')
my_rough_data_df = my_rough_data_pq.read().to_pandas()

In [4]:
my_rough_data_df['date'] = my_rough_data_df['time'].dt.date

In [5]:
my_rough_data_df['forward_diff'] = -my_rough_data_df['time'].diff(periods=-1)

In [6]:
my_rough_data_df['in_day_time_diff'] = my_rough_data_df.groupby('date')['time'].diff().dt.total_seconds() / 3600

In [9]:
my_rough_data_df['in_day_time_diff'].value_counts().sort_index()

in_day_time_diff
1.0     48829
2.0        43
3.0        12
4.0         3
5.0         2
6.0         4
7.0         1
10.0        1
14.0        3
Name: count, dtype: int64

In [ ]:
maintainer_4 = PreRun(4, './temp_fix', None, systems_cleaned)

                     time    energy
0     2007-08-27 12:00:00  0.188190
1     2007-08-27 14:00:00  0.024962
2     2007-08-27 15:00:00  0.035136
3     2007-08-27 16:00:00  0.039177
4     2007-08-27 17:00:00  0.037669
...                   ...       ...
53489 2023-02-27 13:00:00  0.366943
53490 2023-02-27 14:00:00  0.297565
53491 2023-02-27 15:00:00  0.348322
53492 2023-02-27 16:00:00  0.188479
53493 2023-02-27 17:00:00  0.017879

[53494 rows x 2 columns]


In [ ]:
df_temp = maintainer_4.data.copy(deep=True)

In [ ]:
maintainer_4.data.loc[65:75]

,time,energy
65,2007-08-31 16:00:00,0.057845
66,2007-08-31 17:00:00,0.002598
67,2007-09-01 06:00:00,0.032494
68,2007-09-01 07:00:00,0.159633
69,2007-09-01 08:00:00,0.304443
70,2007-09-01 09:00:00,0.265836
71,2007-09-01 10:00:00,0.804245
72,2007-09-01 11:00:00,0.259111
73,2007-09-01 12:00:00,0.161788
74,2007-09-01 13:00:00,0.704303


In [5]:
maintainer_4.add_energy_features_only(
    daily_lags=2,
    remove_daily_lags_nans=False,
    include_last_year=True,
    remove_last_year_nans=False,
    remove_todays_lags_nans=False,
)

AttributeError: module 'pandas' has no attribute 'daterange'

Sidebar for good_runs

In [4]:
good_days_temp = pd.read_csv('./temp_fix/good_days/4_good_days_other.csv')
len(good_days_temp)

4596

In [9]:
good_days_temp_set = set(good_days_temp['date'].unique())
good_days_retry_set = set(good_days_retry['date'].unique())
good_days_retry_set.difference(good_days_temp_set)

{'2010-01-22', '2023-02-28'}

In [13]:
good_days_temp['date'] = pd.to_datetime(good_days_temp['date']).dt.date
good_days_retry['date'] = pd.to_datetime(good_days_retry['date']).dt.date

In [7]:
good_days_retry = pd.read_csv('./try_again_4/good_days/4_good_days_other.csv')

In [8]:
set(good_days_temp['date'].unique()).difference(set(good_days_retry['date'].unique()))  

{'2014-12-04',
 '2011-12-01',
 '2011-12-14',
 '2008-08-28',
 '2022-11-05',
 '2022-12-29',
 '2018-11-20',
 '2011-09-14',
 '2013-03-04',
 '2013-05-02',
 '2015-06-28',
 '2009-05-02',
 '2010-10-11',
 '2020-12-01',
 '2019-06-03',
 '2022-12-07',
 '2022-03-09',
 '2008-01-07',
 '2014-12-29',
 '2015-04-27',
 '2007-08-27',
 '2022-11-18',
 '2020-06-21',
 '2019-03-28',
 '2011-05-23',
 '2016-08-11',
 '2012-06-02',
 '2011-11-28',
 '2021-01-10',
 '2020-07-25',
 '2008-06-26',
 '2016-04-17',
 '2016-11-18',
 '2009-05-01',
 '2022-02-03',
 '2012-11-14',
 '2014-10-16',
 '2012-12-06',
 '2023-02-23',
 '2014-05-30',
 '2019-06-07',
 '2007-12-29',
 '2015-08-31',
 '2018-04-21',
 '2011-07-13',
 '2012-12-09',
 '2015-06-07',
 '2019-07-01',
 '2013-12-20',
 '2018-11-09',
 '2019-06-05',
 '2015-05-07',
 '2011-02-05',
 '2007-09-02',
 '2019-07-20',
 '2010-05-14',
 '2019-08-15',
 '2013-02-08',
 '2007-11-22',
 '2015-06-24',
 '2008-11-03',
 '2015-11-11',
 '2013-06-23',
 '2021-07-14',
 '2007-10-07',
 '2015-07-13',
 '2019-06-

In [15]:
def good_runs_series(dates_ser):
    runs_list = []
    num_dates = len(dates_ser)
    if num_dates == 0:
        return []
    elif num_dates == 1:
        return [[dates_ser.at[0],],]
    else:
        dates_ser = dates_ser.sort_values()
        dates_df = pd.DataFrame(dates_ser)
        dates_df.loc[:, 'forward_diff'] = -dates_df['date'].diff(periods=-1)
        dates_df.loc[:, 'good_forward'] = dates_df.apply(
            lambda row: row['forward_diff'] == pd.Timedelta(days=1),
            axis=1
        )
        end_runs = dates_df[~dates_df['good_forward']]
        num_runs = end_runs.shape[0]
        for j in range(num_runs):
            if j == 0:
                start_ind = 0
            else:
                start_ind = end_runs.index[j-1] + 1
            end_ind = end_runs.index[j]
            jth_run = dates_df.loc[start_ind:end_ind]
            runs_list.append(sorted(jth_run['date']))
        return runs_list

In [23]:
def good_runs_reader(dates_ser):
    run_lengths = []
    good_runs = good_runs_series(dates_ser)
    for run in good_runs:
        print(f'From {run[0]} to {run[-1]}, a run of {len(run)} days.')
        run_lengths.append(len(run))
    run_lengths = pd.Series(run_lengths, name='run_len')
    return run_lengths

In [24]:
loose_runs = good_runs_reader(good_days_temp['date'])

From 2007-08-27 to 2007-11-20, a run of 86 days.
From 2007-11-22 to 2007-12-04, a run of 13 days.
From 2007-12-20 to 2008-03-01, a run of 73 days.
From 2008-03-03 to 2008-04-02, a run of 31 days.
From 2008-04-07 to 2008-08-21, a run of 137 days.
From 2008-08-28 to 2008-11-29, a run of 94 days.
From 2008-12-01 to 2008-12-03, a run of 3 days.
From 2008-12-05 to 2009-01-25, a run of 52 days.
From 2009-01-27 to 2009-04-03, a run of 67 days.
From 2009-04-05 to 2009-04-16, a run of 12 days.
From 2009-04-19 to 2009-05-29, a run of 41 days.
From 2010-01-23 to 2010-01-31, a run of 9 days.
From 2010-02-25 to 2010-03-26, a run of 30 days.
From 2010-03-30 to 2010-04-01, a run of 3 days.
From 2010-04-06 to 2010-04-09, a run of 4 days.
From 2010-04-12 to 2010-04-30, a run of 19 days.
From 2010-05-04 to 2010-05-10, a run of 7 days.
From 2010-05-13 to 2010-06-24, a run of 43 days.
From 2010-06-29 to 2010-07-09, a run of 11 days.
From 2010-07-14 to 2010-07-15, a run of 2 days.
From 2010-10-11 to 2011-0

In [28]:
loose_runs.value_counts().sort_index().tail(15)

run_len
94     1
100    1
112    1
128    1
137    1
147    1
181    1
185    1
203    1
229    1
241    1
254    1
285    1
289    1
292    1
Name: count, dtype: int64

In [87]:
tight_runs = good_runs_reader(good_days_retry['date'])

From 2007-09-01 to 2007-09-01, a run of 1 days.
From 2007-09-03 to 2007-09-04, a run of 2 days.
From 2007-09-07 to 2007-09-08, a run of 2 days.
From 2007-09-11 to 2007-09-15, a run of 5 days.
From 2007-09-18 to 2007-09-19, a run of 2 days.
From 2007-09-21 to 2007-09-23, a run of 3 days.
From 2007-09-25 to 2007-10-06, a run of 12 days.
From 2007-10-08 to 2007-10-12, a run of 5 days.
From 2007-10-15 to 2007-10-20, a run of 6 days.
From 2007-10-22 to 2007-11-16, a run of 26 days.
From 2007-11-18 to 2007-11-19, a run of 2 days.
From 2007-11-24 to 2007-11-27, a run of 4 days.
From 2007-11-29 to 2007-11-30, a run of 2 days.
From 2007-12-02 to 2007-12-03, a run of 2 days.
From 2007-12-20 to 2007-12-20, a run of 1 days.
From 2007-12-22 to 2007-12-24, a run of 3 days.
From 2007-12-30 to 2007-12-31, a run of 2 days.
From 2008-01-02 to 2008-01-06, a run of 5 days.
From 2008-01-09 to 2008-01-10, a run of 2 days.
From 2008-01-12 to 2008-01-14, a run of 3 days.
From 2008-01-17 to 2008-01-17, a run o

In [27]:
tight_runs.value_counts().sort_index().tail(15)

run_len
14    8
15    5
16    6
17    4
18    5
19    1
20    4
22    3
23    3
25    2
26    1
31    2
33    1
38    1
39    1
Name: count, dtype: int64

In [33]:
df_ext = maintainer_4.data.copy(deep=True)

In [5]:
df_ext['time'].dtype

dtype('<M8[ns]')

In [6]:
df_ext['time'] = df_ext['time'].dt.tz_localize('Etc/GMT+7')

In [7]:
first_date = df_ext.at[0, 'time'].date()

In [8]:
first_date

datetime.date(2007, 8, 27)

In [9]:
last_date = df_ext.at[53493, 'time'].date()

In [10]:
df_ext['hour'] = df_ext['time'].dt.hour
df_ext['date'] = df_ext['time'].dt.date
good_dates = list(df_ext['date'].unique())
good_dates.sort()

In [24]:
df_ext['hour'].value_counts(dropna=False)

hour
12    4557
13    4545
11    4539
14    4536
10    4519
15    4480
9     4480
8     4398
16    4324
7     4175
6     3045
17    2851
18    1508
5     1392
19     103
4       10
23       8
0        5
21       4
20       3
22       3
1        3
2        3
3        3
Name: count, dtype: int64

In [25]:
first_date = good_dates[0]
last_date = good_dates[-1]

In [26]:
last_date

datetime.date(2023, 2, 27)

In [115]:
maintainer_4 = PreRun(4, './temp_fix', None, systems_cleaned)
maintainer_4.fill_missing_hours(0)
maintainer_4.add_energy_features_only(daily_lags=3, remove_daily_lags_nans=True, todays_lags=1, remove_todays_lags_nans=False, remove_last_year_nans=False, include_day_of_year_cyclic=True)

                     time    energy
0     2007-08-27 12:00:00  0.188190
1     2007-08-27 14:00:00  0.024962
2     2007-08-27 15:00:00  0.035136
3     2007-08-27 16:00:00  0.039177
4     2007-08-27 17:00:00  0.037669
...                   ...       ...
53489 2023-02-27 13:00:00  0.366943
53490 2023-02-27 14:00:00  0.297565
53491 2023-02-27 15:00:00  0.348322
53492 2023-02-27 16:00:00  0.188479
53493 2023-02-27 17:00:00  0.017879

[53494 rows x 2 columns]


,time,energy,day,1_days_ago,2_days_ago,3_days_ago,1_hours_ago_today,day_of_year_sin,day_of_year_cos
0,2007-11-07 00:00:00,0.0,2007-11-07,0.0,0.0,0.0,NaN,-0.811539,0.584298
1,2007-11-07 01:00:00,0.0,2007-11-07,0.0,0.0,0.0,0.0,-0.811539,0.584298
2,2007-11-07 02:00:00,0.0,2007-11-07,0.0,0.0,0.0,0.0,-0.811539,0.584298
3,2007-11-07 03:00:00,0.0,2007-11-07,0.0,0.0,0.0,0.0,-0.811539,0.584298
4,2007-11-07 04:00:00,0.0,2007-11-07,0.0,0.0,0.0,0.0,-0.811539,0.584298
...,...,...,...,...,...,...,...,...,...
108571,2023-02-27 19:00:00,0.0,2023-02-27,0.0,0.0,0.0,0.0,0.831171,0.556017
108572,2023-02-27 20:00:00,0.0,2023-02-27,0.0,0.0,0.0,0.0,0.831171,0.556017
108573,2023-02-27 21:00:00,0.0,2023-02-27,0.0,0.0,0.0,0.0,0.831171,0.556017
108574,2023-02-27 22:00:00,0.0,2023-02-27,0.0,0.0,0.0,0.0,0.831171,0.556017


In [113]:
maintainer_4.amended_data.iloc[50:60]

,time,energy,1_days_ago,2_days_ago,3_days_ago,1_hours_ago_today,day_of_year_sin,day_of_year_cos
50,2007-08-30 13:00:00,0.828614,NaN,NaN,NaN,0.861318,-0.845249,-0.534373
51,2007-08-30 14:00:00,0.381335,NaN,NaN,NaN,0.828614,-0.845249,-0.534373
52,2007-08-30 15:00:00,0.466359,NaN,NaN,NaN,0.381335,-0.845249,-0.534373
53,2007-08-30 16:00:00,0.110435,NaN,NaN,NaN,0.466359,-0.845249,-0.534373
54,2007-08-30 17:00:00,0.100978,NaN,NaN,NaN,0.110435,-0.845249,-0.534373
55,2007-08-31 06:00:00,0.036473,NaN,NaN,NaN,NaN,-0.854322,-0.519744
56,2007-08-31 07:00:00,0.227354,NaN,NaN,NaN,0.036473,-0.854322,-0.519744
57,2007-08-31 08:00:00,0.458783,NaN,NaN,NaN,0.227354,-0.854322,-0.519744
58,2007-08-31 09:00:00,0.641724,NaN,NaN,NaN,0.458783,-0.854322,-0.519744
59,2007-08-31 10:00:00,0.782232,NaN,NaN,NaN,0.641724,-0.854322,-0.519744


In [107]:
maintainer_4.add_weather_features_only()

In [108]:
maintainer_4.amended_data[['time', 'global_tilted_irradiance']].iloc[0:20]

,time,global_tilted_irradiance
0,2007-08-27 00:00:00,0.000000
1,2007-08-27 01:00:00,0.000000
2,2007-08-27 02:00:00,0.000000
3,2007-08-27 03:00:00,0.000000
4,2007-08-27 04:00:00,0.000000
5,2007-08-27 05:00:00,0.000000
6,2007-08-27 06:00:00,7.415111
7,2007-08-27 07:00:00,103.396584
8,2007-08-27 08:00:00,260.044617
9,2007-08-27 09:00:00,446.956970


In [58]:
df_alt = maintainer_4.amended_data

In [59]:
df_alt['time'].iloc[10:20]

10   2008-04-27 11:00:00
11   2008-04-27 12:00:00
12   2008-04-27 13:00:00
13   2008-04-27 14:00:00
14   2008-04-27 15:00:00
15   2008-04-28 10:00:00
16   2008-04-28 11:00:00
17   2008-04-28 12:00:00
18   2008-04-28 13:00:00
19   2008-04-28 14:00:00
Name: time, dtype: datetime64[ns]

In [64]:
(df_alt['energy'].shift(5) - df_alt['energy']).iloc[10:30]

10   -0.665120
11   -0.347420
12   -0.586441
13   -0.190517
14    0.274686
15    0.720866
16    0.677961
17    0.290237
18   -0.134023
19   -0.218939
20   -0.290716
21   -9.822985
22   -1.665314
23    0.849888
24    0.562392
25    0.117735
26    9.648816
27    2.179393
28   -0.246763
29   -0.150788
Name: energy, dtype: float64

In [11]:
fixed_index = pd.date_range(
    start=pd.Timestamp(year=first_date.year, month=first_date.month, day=first_date.day, hour=0, minute=0, second=0, tz='Etc/GMT+7'),
    end=pd.Timestamp(year=last_date.year, month=last_date.month, day=last_date.day, hour=23, minute=0, second=0, tz='Etc/GMT+7'),
    freq='h'
)

In [12]:
df_fill_na = df_ext.set_index('time').reindex(fixed_index, fill_value=np.nan)

In [15]:
df_fill_na.loc[:, 'energy_alt'] = df_fill_na['energy'].copy(deep=True)

In [14]:
df_fill_na

,energy,hour,date
2007-08-27 00:00:00-07:00,NaN,NaN,NaN
2007-08-27 01:00:00-07:00,NaN,NaN,NaN
2007-08-27 02:00:00-07:00,NaN,NaN,NaN
2007-08-27 03:00:00-07:00,NaN,NaN,NaN
2007-08-27 04:00:00-07:00,NaN,NaN,NaN
...,...,...,...
2023-02-27 19:00:00-07:00,NaN,NaN,NaN
2023-02-27 20:00:00-07:00,NaN,NaN,NaN
2023-02-27 21:00:00-07:00,NaN,NaN,NaN
2023-02-27 22:00:00-07:00,NaN,NaN,NaN


In [19]:
for the_date in df_fill_na['date'].unique():
    df_date = df_fill_na[df_fill_na['date'] == the_date]
    df_date['energy_alt'] = df_date['energy_alt'].interpolate(method='time', limit=2, limit_area='inside')
    df_fill_na.loc[df_date.index, 'energy_alt'] = df_date['energy_alt']

In [22]:
sample_timestamp = pd.Timestamp(year=2026, month=5, day=4, hour=15, tz='ETC/GMT+4')

In [23]:
sample_timestamp.tz_convert('Etc/GMT+5')

Timestamp('2026-05-04 14:00:00-0500', tz='Etc/GMT+5')

In [21]:
df_fill_na.iloc[10:20]

,energy,hour,date,energy_alt
2007-08-27 10:00:00-07:00,NaN,NaN,NaN,NaN
2007-08-27 11:00:00-07:00,NaN,NaN,NaN,NaN
2007-08-27 12:00:00-07:00,0.188190,12.0,2007-08-27,0.188190
2007-08-27 13:00:00-07:00,NaN,NaN,NaN,NaN
2007-08-27 14:00:00-07:00,0.024962,14.0,2007-08-27,0.024962
2007-08-27 15:00:00-07:00,0.035136,15.0,2007-08-27,0.035136
2007-08-27 16:00:00-07:00,0.039177,16.0,2007-08-27,0.039177
2007-08-27 17:00:00-07:00,0.037669,17.0,2007-08-27,0.037669
2007-08-27 18:00:00-07:00,0.033000,18.0,2007-08-27,0.033000
2007-08-27 19:00:00-07:00,0.018578,19.0,2007-08-27,0.018578


In [31]:
df_ext_alt = df_ext.set_index('time')

In [33]:
df_ext_alt = df_ext_alt.interpolate(method='time', limit=2)

In [34]:
df_ext_alt.head()

,energy
time,
2007-08-27 12:00:00-07:00,0.188190
2007-08-27 14:00:00-07:00,0.024962
2007-08-27 15:00:00-07:00,0.035136
2007-08-27 16:00:00-07:00,0.039177
2007-08-27 17:00:00-07:00,0.037669


In [17]:
df_fill_na.iloc[10:20]

,energy,hour,date
2007-08-27 10:00:00-07:00,0.000000,0,0
2007-08-27 11:00:00-07:00,0.000000,0,0
2007-08-27 12:00:00-07:00,0.188190,12,2007-08-27
2007-08-27 13:00:00-07:00,0.000000,0,0
2007-08-27 14:00:00-07:00,0.024962,14,2007-08-27
2007-08-27 15:00:00-07:00,0.035136,15,2007-08-27
2007-08-27 16:00:00-07:00,0.039177,16,2007-08-27
2007-08-27 17:00:00-07:00,0.037669,17,2007-08-27
2007-08-27 18:00:00-07:00,0.033000,18,2007-08-27
2007-08-27 19:00:00-07:00,0.018578,19,2007-08-27


In [12]:
# interpolate short gaps
df_fill_na['energy_ext'] = df_fill_na['energy'].interpolate(method='time', limit=2, limit_area='inside')

In [13]:
df_fill_na.iloc[40:50]

,energy,energy_ext
2007-08-28 16:00:00-07:00,0.283550,0.283550
2007-08-28 17:00:00-07:00,0.088194,0.088194
2007-08-28 18:00:00-07:00,0.003795,0.003795
2007-08-28 19:00:00-07:00,NaN,0.005690
2007-08-28 20:00:00-07:00,NaN,0.007585
2007-08-28 21:00:00-07:00,NaN,NaN
2007-08-28 22:00:00-07:00,NaN,NaN
2007-08-28 23:00:00-07:00,NaN,NaN
2007-08-29 00:00:00-07:00,NaN,NaN
2007-08-29 01:00:00-07:00,NaN,NaN


In [15]:
df_fill_na.iloc[40:50]

,energy,hour,date
2007-08-28 16:00:00-07:00,0.283550,16.0,2007-08-28
2007-08-28 17:00:00-07:00,0.088194,17.0,2007-08-28
2007-08-28 18:00:00-07:00,0.003795,18.0,2007-08-28
2007-08-28 19:00:00-07:00,0.005690,NaN,NaN
2007-08-28 20:00:00-07:00,0.007585,NaN,NaN
2007-08-28 21:00:00-07:00,NaN,NaN,NaN
2007-08-28 22:00:00-07:00,NaN,NaN,NaN
2007-08-28 23:00:00-07:00,NaN,NaN,NaN
2007-08-29 00:00:00-07:00,NaN,NaN,NaN
2007-08-29 01:00:00-07:00,NaN,NaN,NaN


In [ ]:
maintainer_4.data

In [90]:
maintainer_4.amended_data.head()

,time,energy
0,2007-08-27 00:00:00,0.0
1,2007-08-27 01:00:00,0.0
2,2007-08-27 02:00:00,0.0
3,2007-08-27 03:00:00,0.0
4,2007-08-27 04:00:00,0.0


In [ ]:
maintainer_4.amended_data = maintainer_4.data.copy(deep=True)

In [ ]:
maintainer_4.amended_data = maintainer_4.amended_data.set_index('time')

In [ ]:
maintainer_4.amended_data['energy'] = maintainer_4.amended_data['energy'].interpolate(limit=2)

In [ ]:
data_with_lags = maintainer_4.add_energy_features_only(daily_lags=14, include_last_year=True)

In [ ]:
data_good_ends = maintainer_4.good_end_days_naive(14)

In [ ]:
data_good_ends.columns

In [ ]:
data_good_ends.head()

In [ ]:
len(data_good_ends)